# AlexNet Kernel Restriction Study — Phase 2 (FP32 & QAT INT8)

**Author:** Rafael  
**Dataset:** Tiny-ImageNet-200 (200 classes, 64×64 RGB)  
**Goal:** Train and compare all Phase 2 kernel-restriction variants in FP32, then apply
Quantization-Aware Training (QAT) to obtain INT8 versions. Report **Top-1** and **Top-5**
accuracy for every model.

| Model | Kernel strategy | Notes |
|---|---|---|
| AlexNetTV | 11×11 → 5×5 → 3×3 (original) | Pretrained torchvision AlexNet; reference baseline |
| AlexNet3x3FC | All 3×3, FC head | From-scratch; same AlexNet head |
| AlexNet3x3GAP | All 3×3, GAP head | Same 3×3 backbone, GAP head (isolates head type) |
| AlexNet2x2GAP | All 2×2, GAP head | From-scratch; smallest even kernel; no Winograd |
| AlexNet2x2FC | All 2×2, FC head | Same 2×2 backbone, FC head (isolates head type) |
| AlexNetStacked | Stacked 3×3 pairs + BN | Conv-BN-ReLU triples; recovers 5×5 receptive field |
| AlexNetMixed | Alternating 3×3 and 2×2 | Heterogeneous kernel restriction |
| AlexNetSmallKernel | All 3×3, GlobalAvgPool | Lightweight: 1.6M params, single-linear head |

## Notebook layout

1. Imports & reproducibility
2. Dataset & loaders
3. Model shape check
4. Model registration (fuse maps)
5. FP32 training
6. QAT training
7. INT8 conversion & CPU evaluation
8. FP32 evaluation (reload best checkpoints)
9. Final comparison table
10. Persist experiment summary

## 1. Configuration & reproducibility

Everything that controls the experiment lives in a single `ExperimentConfig`
dict. The seed is fixed and `set_seed` propagates it to `random`, `numpy`,
and `torch` (CPU & CUDA).


In [ ]:
import json
import os
import random
import sys
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchinfo
import wandb

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from ml import (
    DataConfig, TrainerConfig, QATConfig,
    create_imagenet_loaders,
    MODEL_REGISTRY, register_model,
    Trainer, make_qat_callback,
    build_qat, build_comparison_table,
    load_best_model, convert_to_int8,
    auto_resume_path, compress_checkpoint,
    create_results_summary, disk_mb, gzip_mb,
    compute_flops, make_run_summary,
)
from configs.loader import load_config

from models import (
    AlexNetTV,
    AlexNet3x3FC,
    AlexNet3x3GAP,
    AlexNet2x2GAP,
    AlexNet2x2FC,
    AlexNetStacked,
    AlexNetMixed,
    AlexNetSmallKernel,
)

torch.backends.quantized.engine = "fbgemm"

In [2]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR    = project_root / "checkpoints" / "phase_2_kernel_restriction_training"
RESULTS_DIR = project_root / "results" / "phase_2_kernel_restriction_training"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

data_cfg = DataConfig(**load_config("data.yaml"))
data_cfg.seed = SEED
fp32_cfg = TrainerConfig(**load_config("training.yaml"))
qat_cfg  = QATConfig(**load_config("qat.yaml"))
print(device)

cuda


## 2. Dataset & loaders

We use Tiny-ImageNet-200 from Kaggle. The train folder is split 90/10 into
train/val with a **seeded generator** so the split is identical on every run.

* Training transforms: light augmentation (random crop + hflip + color jitter)
* Validation transforms: deterministic resize + center crop

Note that we build two `ImageFolder` objects (one per transform) and index
them with the same `Subset` indices — this keeps train-time augmentation
separate from val-time deterministic preprocessing.


In [3]:
import kagglehub

dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")
data_cfg.dataset_path = train_path
print("Tiny-ImageNet train path:", train_path)

Tiny-ImageNet train path: /home/rafael/.cache/kagglehub/datasets/akash2sharma/tiny-imagenet/versions/1/tiny-imagenet-200/train


In [4]:
train_ds, val_ds, train_loader, val_loader = create_imagenet_loaders(data_cfg)

print(f"Train samples: {len(train_ds):,}")
print(f"Val   samples: {len(val_ds):,}")
print(f"Classes:       {data_cfg.num_classes}")
print(f"Batches/epoch: train={len(train_loader)}, val={len(val_loader)}")

Train samples: 90,000
Val   samples: 10,000
Classes:       200
Batches/epoch: train=1406, val=157


## 3. Model shape check

Eight Phase 2 variants — all from `models/alexnet_variants.py`:

| Name | Kernel sizes | BN? | Head |
|---|---|---|---|
| AlexNetTV | 11×11, 5×5, 3×3 (original) | No | 3-layer MLP |
| AlexNet3x3FC | All 3×3 | No | 3-layer MLP |
| AlexNet3x3GAP | All 3×3 | No | Linear(256, 200) |
| AlexNet2x2GAP | All 2×2 | No | Linear(256, 200) |
| AlexNet2x2FC | All 2×2 | No | 3-layer MLP |
| AlexNetStacked | Stacked 3×3 pairs | **Yes** | 3-layer MLP |
| AlexNetMixed | Alternating 3×3/2×2 | No | Linear(256, 200) |
| AlexNetSmallKernel | All 3×3, GAP | No | Linear(256, 200) |

In [5]:
x = torch.randn(2, 3, data_cfg.img_size, data_cfg.img_size)
for label, ctor in [
    ("AlexNetTV",          AlexNetTV),
    ("AlexNet3x3FC",       AlexNet3x3FC),
    ("AlexNet3x3GAP",      AlexNet3x3GAP),
    ("AlexNet2x2GAP",      AlexNet2x2GAP),
    ("AlexNet2x2FC",       AlexNet2x2FC),
    ("AlexNetStacked",     AlexNetStacked),
    ("AlexNetMixed",       AlexNetMixed),
    ("AlexNetSmallKernel", AlexNetSmallKernel),
]:
    m = ctor().eval()
    with torch.no_grad():
        y = m(x)
    assert y.shape == (2, data_cfg.num_classes), f"{label}: unexpected shape {y.shape}"
    info = torchinfo.summary(m, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    print(f"{label:20s} OK -> {tuple(y.shape)}, params={info.total_params/1e6:.2f}M")

AlexNetTV            OK -> (2, 200), params=57.82M
AlexNet3x3FC         OK -> (2, 200), params=57.61M
AlexNet3x3GAP        OK -> (2, 200), params=2.30M
AlexNet2x2GAP        OK -> (2, 200), params=1.05M
AlexNet2x2FC         OK -> (2, 200), params=35.38M
AlexNetStacked       OK -> (2, 200), params=60.48M
AlexNetMixed         OK -> (2, 200), params=1.75M
AlexNetSmallKernel   OK -> (2, 200), params=1.60M


## 4. Model registration

Fuse maps are index-based paths inside `features` for flat-Sequential models.
`AlexNetStacked` uses Conv-BN-ReLU triples; all others use Conv-ReLU pairs.
All eight models support QAT — no skip set needed.

In [6]:
# Conv-ReLU pairs (no BN) — shared by AlexNetTV, 3x3FC/GAP, 2x2GAP/FC, Mixed, SmallKernel
FUSE_CONV_RELU = [["0","1"],["3","4"],["6","7"],["8","9"],["10","11"]]

# Conv-BN-ReLU triples — AlexNetStacked (10 conv layers, 5 stages × 2 blocks)
FUSE_MAP_STACKED = [
    ["0","1","2"],["3","4","5"],      # Stage 1 (MaxPool at 6)
    ["7","8","9"],["10","11","12"],   # Stage 2 (MaxPool at 13)
    ["14","15","16"],["17","18","19"],# Stage 3
    ["20","21","22"],["23","24","25"],# Stage 4
    ["26","27","28"],["29","30","31"],# Stage 5
]

MODEL_REGISTRY.clear()
register_model("alexnet_tv",           AlexNetTV,           fuse_map=FUSE_CONV_RELU,   fuse_root_attr="features", lr=3e-4)
register_model("alexnet_3x3_fc",       AlexNet3x3FC,        fuse_map=FUSE_CONV_RELU,   fuse_root_attr="features", lr=3e-4)
register_model("alexnet_3x3_gap",      AlexNet3x3GAP,       fuse_map=FUSE_CONV_RELU,   fuse_root_attr="features", lr=3e-4)
register_model("alexnet_2x2_gap",      AlexNet2x2GAP,       fuse_map=FUSE_CONV_RELU,   fuse_root_attr="features", lr=3e-4)
register_model("alexnet_2x2_fc",       AlexNet2x2FC,        fuse_map=FUSE_CONV_RELU,   fuse_root_attr="features", lr=3e-4)
register_model("alexnet_stacked",      AlexNetStacked,      fuse_map=FUSE_MAP_STACKED, fuse_root_attr="features", lr=1e-3)
register_model("alexnet_mixed",        AlexNetMixed,        fuse_map=FUSE_CONV_RELU,   fuse_root_attr="features", lr=3e-4)
register_model("alexnet_small_kernel", AlexNetSmallKernel,  fuse_map=FUSE_CONV_RELU,   fuse_root_attr="features", lr=3e-4)

print("Registered:", list(MODEL_REGISTRY))

Registered: ['alexnet_tv', 'alexnet_3x3_fc', 'alexnet_3x3_gap', 'alexnet_2x2_gap', 'alexnet_2x2_fc', 'alexnet_stacked', 'alexnet_mixed', 'alexnet_small_kernel']


## 5. FP32 training

In [7]:
fp32_training_results = {}

for name, spec in MODEL_REGISTRY.items():
    # Auto-detect resume checkpoint
    resume_from = auto_resume_path(SAVE_DIR, name)
    
    # Restore W&B run if resuming
    run_id = None
    if resume_from:
        meta_path = SAVE_DIR / f"{name}_meta.json"
        if meta_path.exists():
            run_id = json.loads(meta_path.read_text()).get("wandb_run_id")
    
    model_cfg = replace(fp32_cfg, lr=spec.get("lr", fp32_cfg.lr))
    print("=" * 72)
    print(f"Training: {name}  lr={model_cfg.lr}  epochs={model_cfg.epochs}")
    if resume_from:
        print(f"(Resuming from checkpoint)")
    print("=" * 72)

    model = spec["ctor"]().to(device)

    run = wandb.init(
        project="alexnet-phase2",
        group="fp32-phase2",
        name=f"{name}_fp32",
        id=run_id,
        resume="allow" if run_id else None,
        config={**asdict(model_cfg), "arch": name, "phase": "fp32",
                "num_classes": data_cfg.num_classes, "img_size": data_cfg.img_size,
                "dataset": "tiny-imagenet-200"},
        tags=["phase2", "kernel-restriction", "tiny-imagenet", "fp32"],
        mode="offline",
    )

    trainer = Trainer(
        model, train_loader, val_loader, model_cfg,
        device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
        wandb_run=run,
        log_file=SAVE_DIR / f"{name}.log",
    )
    results = trainer.fit(resume_from=resume_from)
    wandb.finish()

    fp32_training_results[name] = results

    del model
    torch.cuda.empty_cache()

print("\nFP32 training complete.")

Training: alexnet_tv  lr=0.0003  epochs=100
(Resuming from checkpoint)


wandb: WARNING `resume` will be ignored since W&B syncing is set to `offline`. Starting a new run with run id cqp23xoa.


Validation: 100%|██████████| 157/157 [00:01<00:00, 117.01it/s, loss=3.7673, top1=24.48%, top5=49.09%]
wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch  20/100 | train_loss=3.9681 train_acc=20.12% | val_loss=3.7673 val_acc=24.48% val_top5=49.09% | lr=2.71e-04 peak_mem=1958MB time=48.5s
Validation: 100%|██████████| 157/157 [00:01<00:00, 116.50it/s, loss=3.7512, top1=24.22%, top5=49.35%]
Epoch  21/100 | train_loss=3.9630 train_acc=20.24% | val_loss=3.7512 val_acc=24.22% val_top5=49.35% | lr=2.69e-04 peak_mem=1958MB time=48.4s
Validation: 100%|██████████| 157/157 [00:01<00:00, 144.98it/s, loss=3.7620, top1=24.37%, top5=48.60%]
Epoch  22/100 | train_loss=3.9557 train_acc=20.41% | val_loss=3.7620 val_acc=24.37% val_top5=48.60% | lr=2.66e-04 peak_mem=1958MB time=48.1s

best_val_acc,▁▂▂▃▃▃▃▃▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇█
epoch_time_s,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁
lr,█████▇▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁
peak_gpu_mem_mb,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_acc,▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇█████
train_loss,███▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
val_acc,▁▁▁▂▂▃▃▃▃▃▃▃▃▃▄▄▅▄▄▅▆▆▅▅▆▆▆▆▆▇▇▇▆▇▇▇█▇██
val_loss,███▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▃▃▃▃▃▃▂▃▂▂▂▁▂▂▁▁
val_top5,▁▂▁▂▃▂▃▄▄▃▃▄▄▄▄▄▅▅▅▅▆▆▅▆▆▆▇▇▆▇▇▇▇█▇█████
best_val_acc,32.26
epoch_time_s,47.88424


Training: alexnet_3x3_fc  lr=0.0003  epochs=100
(Resuming from checkpoint)


...

wandb: WARNING `resume` will be ignored since W&B syncing is set to `offline`. Starting a new run with run id 9w89ebol.


Training: alexnet_small_kernel  lr=0.0003  epochs=100
(Resuming from checkpoint)


Validation: 100%|██████████| 157/157 [00:01<00:00, 93.18it/s, loss=2.9708, top1=45.30%, top5=71.30%] 
Epoch  64/100 | train_loss=2.8615 train_acc=46.55% | val_loss=2.9708 val_acc=45.30% val_top5=71.30% | lr=8.61e-05 peak_mem=1038MB time=24.2s
Early stopping at epoch 64

================= Run Summary =================
Model          : alexnet_small_kernel
Epochs         : 64
Best Val Top-1 : 45.84%
Best Val Top-5 : 70.95%
Final Val Top-1: 45.30%
Final Val Top-5: 71.30%
Best Val Loss  : inf
Total Time     : 00:00:24


epoch_time_s,▁
lr,▁
peak_gpu_mem_mb,▁
train_acc,▁
train_loss,▁
val_acc,▁
val_loss,▁
val_top5,▁
epoch_time_s,24.21106
lr,9e-05
peak_gpu_mem_mb,1038.2085



FP32 training complete.


## 6. Quantization-Aware Training (QAT)

All six models support QAT — fuse maps are defined for every architecture.
QAT trains on GPU; `convert` and INT8 inference run on CPU (fbgemm).

In [8]:
qat_train_cfg = replace(
    fp32_cfg,
    epochs=qat_cfg.epochs,
    lr=qat_cfg.lr,
    weight_decay=qat_cfg.weight_decay,
    use_amp=False,  # AMP incompatible with fake-quant observers
)

qat_models = {}
qat_training_results = {}

for name in MODEL_REGISTRY:
    # Auto-detect resume checkpoint
    resume_from = auto_resume_path(SAVE_DIR, f"qat_{name}")
    
    # Restore W&B run if resuming
    run_id = None
    if resume_from:
        meta_path = SAVE_DIR / f"qat_{name}_meta.json"
        if meta_path.exists():
            run_id = json.loads(meta_path.read_text()).get("wandb_run_id")
    
    print("=" * 72)
    print(f"QAT fine-tuning: {name}")
    if resume_from:
        print(f"(Resuming from checkpoint)")
    print("=" * 72)

    qat_model = build_qat(name, save_dir=SAVE_DIR, device=device)
    cb = make_qat_callback(qat_cfg.freeze_bn_epoch, qat_cfg.disable_observer_epoch)

    run = wandb.init(
        project="alexnet-phase2",
        group="qat-phase2",
        name=f"{name}_qat",
        id=run_id,
        resume="allow" if run_id else None,
        config={
            **asdict(qat_train_cfg),
            "arch": name,
            "phase": "qat",
            "freeze_bn_epoch": qat_cfg.freeze_bn_epoch,
            "disable_observer_epoch": qat_cfg.disable_observer_epoch,
            "num_classes": data_cfg.num_classes, "img_size": data_cfg.img_size,
            "dataset": "tiny-imagenet-200",
        },
        tags=["phase2", "kernel-restriction", "tiny-imagenet", "qat"],
        mode="offline",
    )

    trainer = Trainer(
        qat_model, train_loader, val_loader, qat_train_cfg,
        device, SAVE_DIR, f"qat_{name}", num_classes=data_cfg.num_classes,
        wandb_run=run,
        epoch_callback=cb,
        log_file=SAVE_DIR / f"qat_{name}.log",
    )
    results = trainer.fit(resume_from=resume_from)
    wandb.finish()

    qat_training_results[name] = results
    qat_models[name] = qat_model.cpu()

    del qat_model
    torch.cuda.empty_cache()

print("\nQAT training complete.")

QAT fine-tuning: alexnet_tv
(Resuming from checkpoint)


/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(
wandb: WARNING `resume` will be ignored since W&B syncing is set to `offline`. Starting a new run with run id 5d30j8gh.



================= Run Summary =================
Model          : qat_alexnet_tv
Epochs         : 20
Best Val Top-1 : 26.43%
Best Val Top-5 : 51.77%
Final Val Top-1: 26.25%
Final Val Top-5: 51.73%
Best Val Loss  : inf
Total Time     : 00:00:00


QAT fine-tuning: alexnet_3x3_fc
(Resuming from checkpoint)


/home/rafael/Documents/alexnet_rafael/.venv/lib/python3.12/site-packages/torch/ao/quantization/observer.py:229: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(
wandb: WARNING `resume` will be ignored since W&B syncing is set to `offline`. Starting a new run with run id ubef7rwb.


...

QAT fine-tuning: alexnet_small_kernel
(Resuming from checkpoint)


wandb: WARNING `resume` will be ignored since W&B syncing is set to `offline`. Starting a new run with run id ule9w4md.


Validation: 100%|██████████| 157/157 [00:02<00:00, 70.28it/s, loss=2.9617, top1=46.09%, top5=70.99%]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
Epoch  11/20 | train_loss=2.8375 train_acc=47.48% | val_loss=2.9617 val_acc=46.09% val_top5=70.99% | lr=4.22e-06 peak_mem=1196MB time=47.7s
Validation: 100%|██████████| 157/157 [00:02<00:00, 70.40it/s, loss=2.9566, top1=46.05%, top5=71.13%]
Epoch  12/20 | train_loss=2.8365 train_acc=47.46% | val_loss=2.9566 val_acc=46.05% val_top5=71.13% | lr=3.45e-06 peak_mem=1196MB time=47.8s
Validation: 100%|██████████| 157/157 [00:02<00:00, 69.28it/s, loss=2.9609, top1=45.94%, top5=70.86%]
Epoch  13/20 | train_loss=2.8372 train_acc=47.49% | val_loss=2.9609 val_acc=45.94% val_top5=70.86% | lr=2.73e-06 peak_mem=1196MB time=47.7s
Validation: 100%|██████████| 157/157 [00:02<00:00, 69.19it/s, loss=2.9596, top1=46.01%, top5=71.15%]
Epoch  14/20 | train_loss=2.8329 train_acc=47.59% | val_loss=2.9596 val_acc

best_val_acc,▁▄█
epoch_time_s,▃▅▂▅▆▁▅▅█▅
lr,█▇▆▄▃▃▂▁▁▁
peak_gpu_mem_mb,█▁▁▁▁▁▁▁▁▁
train_acc,▄▄▅█▁▇▃▃▁▇
train_loss,▇▆▇▂▇▁█▆▃▆
val_acc,▆▅▃▄▁▇▅▇█▄
val_loss,█▄▇▆▄▅▆▁▄▃
val_top5,▃▄▁▅▆▅▅█▆▆
best_val_acc,46.22
epoch_time_s,47.76744



QAT training complete.


## 7. INT8 conversion & CPU evaluation

`torch.ao.quantization.convert` materialises true quantised ops and must run on CPU.

In [ ]:
val_loader_cpu = torch.utils.data.DataLoader(
    val_ds, batch_size=data_cfg.batch_size, shuffle=False, num_workers=0, pin_memory=False,
)
cpu_device = torch.device("cpu")

int8_models = {}
for name in MODEL_REGISTRY:
    int8_models[name] = convert_to_int8(qat_models[name].cpu().eval())

for name, m in int8_models.items():
    torch.save(m.state_dict(), SAVE_DIR / f"{name}.pth")
    compress_checkpoint(SAVE_DIR / f"{name}.pth")
print("INT8 conversion done.")

int8_metrics = {}
for name, m in int8_models.items():
    m = m.cpu().eval()
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    trainer = Trainer(m, val_loader_cpu, val_loader_cpu, dummy_cfg, cpu_device, SAVE_DIR, name,
                      num_classes=data_cfg.num_classes)
    metrics = trainer.evaluate(topk=(1, 5))
    int8_metrics[name] = metrics
    print(f"{name:22s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

## 8. FP32 evaluation — reload best checkpoints

In [10]:
CTORS = {
    "alexnet_tv":           AlexNetTV,
    "alexnet_3x3_fc":       AlexNet3x3FC,
    "alexnet_3x3_gap":      AlexNet3x3GAP,
    "alexnet_2x2_gap":      AlexNet2x2GAP,
    "alexnet_2x2_fc":       AlexNet2x2FC,
    "alexnet_stacked":      AlexNetStacked,
    "alexnet_mixed":        AlexNetMixed,
    "alexnet_small_kernel": AlexNetSmallKernel,
}

fp32_best = {name: load_best_model(name, ctor, SAVE_DIR, device) for name, ctor in CTORS.items()}

fp32_metrics = {}
for name, m in fp32_best.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    trainer = Trainer(
        m, train_loader, val_loader, dummy_cfg,
        device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    )
    metrics = trainer.evaluate(topk=(1, 5))
    fp32_metrics[name] = metrics
    print(f"{name:22s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

alexnet_tv             | loss=3.0083 | top1=32.20% | top5=57.61%
alexnet_3x3_fc         | loss=2.8845 | top1=35.79% | top5=61.28%
alexnet_3x3_gap        | loss=2.6121 | top1=40.32% | top5=66.32%
alexnet_2x2_gap        | loss=2.9859 | top1=33.15% | top5=58.95%
alexnet_2x2_fc         | loss=3.1185 | top1=30.53% | top5=56.03%
alexnet_stacked        | loss=2.3538 | top1=45.22% | top5=70.96%
alexnet_mixed          | loss=2.6837 | top1=38.74% | top5=64.83%
alexnet_small_kernel   | loss=2.3493 | top1=45.84% | top5=71.08%


## 9. Final comparison table

In [11]:
rows = []

for name, m in fp32_best.items():
    info = torchinfo.summary(m, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": name, "precision": "FP32",
        "top1_%": fp32_metrics[name]["top1"],
        "top5_%": fp32_metrics[name]["top5"],
        "loss":   fp32_metrics[name]["loss"],
        "params_M": info.total_params / 1e6,
        "size_MB":  disk_mb(SAVE_DIR / f"{name}_best.pth"),
    })

for name, m in int8_models.items():
    info = torchinfo.summary(fp32_best[name], input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": f"{name}_INT8", "precision": "INT8",
        "top1_%": int8_metrics[name]["top1"],
        "top5_%": int8_metrics[name]["top5"],
        "loss":   int8_metrics[name]["loss"],
        "params_M": info.total_params / 1e6,
        "size_MB":  disk_mb(SAVE_DIR / f"{name}.pth"),
    })

df = build_comparison_table(rows)
df.to_csv(RESULTS_DIR / "final_comparison.csv", index=False)
df

,model,precision,top1_%,top5_%,loss,params_M,size_MB
0,alexnet_small_kernel,FP32,45.836565,71.076936,2.349307,1.602376,18.353315
1,alexnet_stacked,FP32,45.219755,70.960724,2.353797,60.483976,692.255035
2,alexnet_3x3_gap,FP32,40.316811,66.320425,2.612064,2.302984,26.371012
3,alexnet_mixed,FP32,38.740057,64.825773,2.683706,1.750024,20.042604
4,alexnet_3x3_fc,FP32,35.787162,61.282206,2.884496,57.605128,659.258051
5,alexnet_2x2_gap,FP32,33.152020,58.946741,2.985930,1.052744,12.063150
6,alexnet_tv,FP32,32.202688,57.607883,3.008257,57.823240,661.754324
7,alexnet_2x2_fc,FP32,30.532345,56.028664,3.118512,35.383368,404.950323
8,alexnet_stacked_INT8,INT8,42.309719,68.345517,2.511703,60.483976,57.941553
9,alexnet_mixed_INT8,INT8,37.993860,63.802063,2.724205,1.750024,1.705048


In [ ]:
import json as _json

# --- Benchmark FP32 models (GPU) ---
fp32_benchmarks = {}
for name, m in fp32_best.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    t = Trainer(m, train_loader, val_loader, dummy_cfg, device, SAVE_DIR, name,
                num_classes=data_cfg.num_classes)
    fp32_benchmarks[name] = t.benchmark()
    print(f"{name:22s} FP32 | {fp32_benchmarks[name]['latency_ms_per_image']:.3f} ms/img")

# --- Benchmark INT8 models (CPU) ---
int8_benchmarks = {}
for name, m in int8_models.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    t = Trainer(m.cpu().eval(), val_loader_cpu, val_loader_cpu, dummy_cfg, cpu_device, SAVE_DIR, name,
                num_classes=data_cfg.num_classes)
    int8_benchmarks[name] = t.benchmark(warmup=50)
    print(f"{name:22s} INT8 | {int8_benchmarks[name]['latency_ms_per_image']:.3f} ms/img")

# --- FLOPs ---
fp32_flops = {}
for name, m in fp32_best.items():
    fp32_flops[name] = compute_flops(m.cpu().eval())
    print(f"{name:22s} | {fp32_flops[name]['macs']/1e9:.3f} GMACs")

# --- Per-model summary JSONs ---
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for name in fp32_best:
    compress_checkpoint(SAVE_DIR / f"{name}_best.pth")
    info = torchinfo.summary(fp32_best[name], input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    summary = make_run_summary(
        name=name,
        mode="fp32+qat+int8",
        fit_results=fp32_training_results.get(name, {}),
        fp32_eval=fp32_metrics[name],
        params_m=info.total_params / 1e6,
        fp32_size_mb=disk_mb(SAVE_DIR / f"{name}_best.pth"),
        int8_size_mb=disk_mb(SAVE_DIR / f"{name}.pth"),
        fp32_benchmark=fp32_benchmarks[name],
        flops_results=fp32_flops[name],
        int8_eval=int8_metrics.get(name),
        int8_benchmark=int8_benchmarks.get(name),
        fp32_gzip_mb=gzip_mb(SAVE_DIR / f"{name}_best.pth"),
        int8_gzip_mb=gzip_mb(SAVE_DIR / f"{name}.pth"),
    )
    out = RESULTS_DIR / f"{name}_summary.json"
    with open(out, "w") as f:
        _json.dump(summary, f, indent=2, default=str)
    print(f"Saved: {out.name}")

In [13]:
print("=" * 72)
print("RANKING BY TOP-1 ACCURACY (all precisions)")
print("=" * 72)
ranked = df.sort_values("top1_%", ascending=False).reset_index(drop=True)
for i, row in ranked.iterrows():
    print(f"{i+1:2d}. {row['model']:22s} [{row['precision']}] | "
          f"top1={row['top1_%']:6.2f}% | top5={row['top5_%']:6.2f}% | "
          f"params={row['params_M']:6.2f}M | size={row['size_MB']:6.2f}MB")


RANKING BY TOP-1 ACCURACY (all precisions)
 1. alexnet_small_kernel   [FP32] | top1= 45.84% | top5= 71.08% | params=  1.60M | size= 18.35MB
 2. alexnet_stacked        [FP32] | top1= 45.22% | top5= 70.96% | params= 60.48M | size=692.26MB
 3. alexnet_stacked_INT8   [INT8] | top1= 42.31% | top5= 68.35% | params= 60.48M | size= 57.94MB
 4. alexnet_3x3_gap        [FP32] | top1= 40.32% | top5= 66.32% | params=  2.30M | size= 26.37MB
 5. alexnet_mixed          [FP32] | top1= 38.74% | top5= 64.83% | params=  1.75M | size= 20.04MB
 6. alexnet_mixed_INT8     [INT8] | top1= 37.99% | top5= 63.80% | params=  1.75M | size=  1.71MB
 7. alexnet_3x3_gap_INT8   [INT8] | top1= 37.02% | top5= 62.89% | params=  2.30M | size=  2.23MB
 8. alexnet_3x3_fc_INT8    [INT8] | top1= 36.19% | top5= 62.16% | params= 57.61M | size= 55.12MB
 9. alexnet_small_kernel_INT8 [INT8] | top1= 36.16% | top5= 61.60% | params=  1.60M | size=  1.56MB
10. alexnet_3x3_fc         [FP32] | top1= 35.79% | top5= 61.28% | params= 57.61M 

## 10. Persist experiment summary

In [14]:
create_results_summary(
    results={
        "fp32_metrics": fp32_metrics,
        "int8_metrics": int8_metrics,
        "fp32_training_results": fp32_training_results,
        "qat_training_results": qat_training_results,
    },
    config=asdict(fp32_cfg),
    output_path=RESULTS_DIR / "experiment_summary.json",
)

## W&B — syncing offline runs

All runs were saved locally with `mode="offline"`. Two run groups are tracked:
- **`fp32-phase2`** — one run per architecture, FP32 training
- **`qat-phase2`** — one run per architecture, QAT fine-tuning

When ready, sync to the W&B dashboard from a terminal:

```bash
wandb sync --sync-all
```